# Material choices under an emissions budget

Size solar generation and battery storage to meet a representative load. Compare the least-cost design with a design subject to a tighter embodied emissions budget. Material use is linked to installed capacity. Demand and storage dispatch are resolved within the day.

The PV and battery material coefficients come from the legacy material case. The example limits thin-film capacity and assigns a small illustrative premium to brine-based storage to make the cost–emissions trade-off visible.

## Inputs

In [1]:
from math import isclose

import pandas as pd
from energia import Material, Model, Periods, Process, Resource, Storage
from gana import inf

demand = [3, 5, 4, 2]
solar = [0.35, 0.75, 0.5, 0]
# PV technology: construction material, material requirement, capacity cost, capacity limit.
pv_options = {
    "mono_pv": ("monocrystalline", 0.4, 993_991, 30),
    "poly_pv": ("polycrystalline", 0.36, 993_991, 30),
    "thin_film_pv": ("cdte", 0.02, 575_720, 3),
}
# Storage technology: construction material, requirement per inventory unit, inventory capacity cost.
storage_options = {
    "rock_battery": ("rock_lithium", 20, 1_343_614),
    "brine_battery": ("brine_lithium", 20, 1_475_000),
}
material_emissions = {
    "monocrystalline": 20, "polycrystalline": 26, "cdte": 40,
    "rock_lithium": 1.484, "brine_lithium": 0.031,
}

## Connect procurement to construction

The material balances account for all installations. Embodied emissions are the sum of purchased material quantities multiplied by the corresponding emissions intensities.

In [2]:
def design_under_cap(emissions_cap=None, minimize_emissions=False):
    m = Model("material_choices")
    m.block = Periods()
    m.day = len(demand) * m.block
    m.sun, m.power = Resource(), Resource()
    _ = m.sun.consume(m.block) >= 0
    served = m.power.release(m.block)
    _ = served == demand
    m.declare(Material, list(material_emissions))

    carbon = 0
    purchases = {}
    for name, intensity in material_emissions.items():
        use = getattr(m, name).consume(m.day)
        _ = use >= 0
        carbon += intensity * use.V()[0]
        purchases[name] = use

    cost = 0
    design = {}
    for name, (material, requirement, price, limit) in pv_options.items():
        plant = Process(m.power == -m.sun)
        setattr(m, name, plant)
        capacity = plant.capacity(m.day)
        _ = capacity <= limit
        _ = plant.operate(m.block) <= solar
        _ = getattr(m, material).use(capacity) == requirement
        plant.locate()
        cost += price * capacity.V()[0]
        design[name] = capacity

    for name, (material, requirement, price) in storage_options.items():
        battery = Storage(m.power == 0.8)
        setattr(m, name, battery)
        inventory_capacity = battery.capacity(m.day)
        _ = inventory_capacity <= 20
        _ = getattr(m, material).use(inventory_capacity) == requirement
        battery.locate()
        cost += price * inventory_capacity.V()[0]
        design[name] = inventory_capacity

    if emissions_cap is not None:
        m.program.embodied_carbon_limit = carbon <= emissions_cap
    m.program.design = inf(carbon if minimize_emissions else cost)
    m.program.opt()

    assert isclose(sum(served.output(aslist=True)), sum(demand), abs_tol=1e-6)
    installed = {name: sample.output(aslist=True)[0] for name, sample in design.items()}
    total_cost = sum(pv_options[name][2] * installed[name] for name in pv_options)
    total_cost += sum(storage_options[name][2] * installed[name] for name in storage_options)
    footprint = sum(
        material_emissions[name] * sample.output(aslist=True)[0]
        for name, sample in purchases.items()
    )
    expected = footprint if minimize_emissions else total_cost
    assert isclose(expected, m.program.active_objective.X, rel_tol=1e-6)
    return {"Cost": total_cost, "Embodied emissions": footprint, **installed}

## Compare the designs

First find the lowest achievable emissions and the emissions of the least-cost design. Set the cap halfway between those values and solve again for cost.

In [3]:
least_cost = design_under_cap()
least_carbon = design_under_cap(minimize_emissions=True)
cap = (least_cost["Embodied emissions"] + least_carbon["Embodied emissions"]) / 2
capped = design_under_cap(emissions_cap=cap)

assert least_carbon["Embodied emissions"] < least_cost["Embodied emissions"]
assert capped["Embodied emissions"] <= cap + 1e-6
assert capped["Cost"] >= least_cost["Cost"] - 1e-6
pd.DataFrame([least_cost, capped], index=["Least cost", "With emissions cap"]).round(2)

,Cost,Embodied emissions,mono_pv,poly_pv,thin_film_pv,rock_battery,brine_battery
Least cost,11112265.44,133.34,0.00,6.06,3.0,2.50,0.00
With emissions cap,11257859.30,92.90,6.06,0.00,3.0,1.39,1.11
